<a href="https://colab.research.google.com/github/22f2001565/System-Threat-Forecaster/blob/main/22f2001565_notebook_t12025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

system_threat_forecaster_path = kagglehub.competition_download('System-Threat-Forecaster')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib .pyplot as plt
import seaborn as sns

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression, SGDClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Data Loading

In [ ]:
train_data=pd.read_csv('/kaggle/input/System-Threat-Forecaster/train.csv')
test_data=pd.read_csv('/kaggle/input/System-Threat-Forecaster/test.csv')

# Exploratory Data Analysis

In [ ]:
train_data.head()
#test_data.head()

In [ ]:
train_data.info()
#test_data.info()

In [ ]:
train_data.isna().sum()
test_data.isna().sum()

In [ ]:
train_data.shape

In [ ]:
test_data.shape

In [ ]:
train_data.describe(include='all').T

In [ ]:
#missing value graph
missing_values = train_data.isnull().sum()
missing_values = missing_values[missing_values > 0]
plt.figure(figsize=(10, 5))
missing_values.sort_values().plot(kind='barh', color='red')
plt.title("Missing Values Count")
plt.show()

1. **High Missing Values in Key Features**  
   - The feature **'SMode'** has the highest number of missing values, approaching 1000. This suggests that nearly all records in the dataset lack values for this feature, making it a candidate for either imputation or removal, depending on its importance to the analysis.
   - Other features such as **'CityID', 'RegionIdentifier', and 'CfcFeatures'** also have a significant proportion of missing values. These could indicate location-based information or system-specific details that may not have been recorded consistently.

2. **Moderate Missing Values in Several Features**  
   - Features like **'InternalBatteryNumberOfCharges', 'FirmwareManufacturerID', 'FirmwareVersionID'**, and **'BrightnessDisable'** have a substantial amount of missing data, but not as extreme as the topmost features.
   - These missing values may be due to inconsistencies in data collection or differences in device configurations.

3. **Low Missing Values in Certain Features**  
   - Features such as **'SystemVolumeCapacityMB', 'ProcessorModelID', 'AntivirusConfigurationID'**, and a few others have relatively fewer missing values.
   - While the percentage of missing values is low, it is still important to consider proper handling techniques to avoid data integrity issues.


In [ ]:
# Distribution of numerical variables
num_cols = train_data.select_dtypes(include=['int64', 'float64']).columns

train_data[num_cols].hist(figsize=(15, 12), bins=30, edgecolor='black')
plt.suptitle("Histograms of Numerical Features")
plt.tight_layout()
plt.show()

1. **Binary Features with Discrete Distributions:**  
   - Features such as **'IsBetaUser'**, **'RealTimeProtectionState'**, **'IsPassiveModeEnabled'**, and **'FirewallEnabled'** display distributions with only a few distinct values (mostly 0s and 1s), indicating that these are binary categorical variables.
   - These variables typically represent system settings or security configurations that either are enabled (1) or disabled (0).

2. **Skewed Distributions in System and Hardware-Related Features:**  
   - Features like **'PrimaryDiskCapacityMB'**, **'SystemVolumeCapacityMB'**, and **'TotalPhysicalRAMMB'** exhibit strong right-skewness, with a majority of values concentrated on the lower end but some extending to significantly high values.
   - This indicates that while most devices have standard storage and RAM configurations, some high-end machines with large capacities create long tails in the distribution.

3. **Categorical Features with Distinct Peaks:**  
   - Features like **'CountryID'**, **'CityID'**, **'GeoRegionID'**, and **'LocaleEnglishNameID'** show distinct peaks, suggesting that certain country and city identifiers are much more common in the dataset.
   - These variables are important for geographic analysis and may help in detecting patterns related to different regions.

4. **Processor and Firmware Features:**  
   - Features like **'ProcessorCoreCount'**, **'ProcessorManufacturerID'**, **'ProcessorModelID'**, and **'FirmwareVersionID'** exhibit a clustered distribution, indicating that certain processor models and manufacturers dominate the dataset.
   - These features are crucial for performance-based analysis and could play a role in predictive modeling.

5. **Resolution and Display Features:**  
   - Features such as **'PrimaryDisplayResolutionHorizontal'** and **'PrimaryDisplayResolutionVertical'** have a few dominant values, likely representing common screen resolutions.
   - These distributions suggest that most devices have standard screen resolutions, with only a few outliers having exceptionally high values.

6. **Target Variable Distribution:**  
   - The histogram for the **'target'** variable shows how the dependent variable is distributed. If it is highly imbalanced, appropriate techniques like oversampling or undersampling might be required for model training.


In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(train_data[num_cols].corr(), annot=False, cmap='coolwarm', linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

1. **Strong Correlations (Red Areas):**  
   - Some features exhibit strong positive correlations, indicating that they are related. For example, **'PrimaryDisplayResolutionVertical'** and **'PrimaryDisplayResolutionHorizontal'** are likely correlated, as both describe aspects of screen resolution.
   - **'ProcessorManufacturerID'** and **'ProcessorModelID'** may also have a high correlation since specific manufacturers produce certain models.

2. **Negative Correlations (Blue Areas):**  
   - Some features show negative correlations, implying an inverse relationship. If a feature is negatively correlated with another, increasing one might mean decreasing the other.
   - **'IsPortableOS'** might be negatively correlated with features related to disk capacity, as portable systems may have lower storage.

3. **Independent Features (Near Zero Correlation):**  
   - Many features appear uncorrelated (closer to white areas), meaning they have little influence on each other.
   - This is expected in datasets containing diverse categorical and numerical system attributes.

4. **Feature Selection Insights:**  
   - Highly correlated features can introduce multicollinearity in machine learning models, which may negatively impact performance.
   - Redundant variables (e.g., features representing similar aspects of the system) can be removed or combined to improve model efficiency.


In [ ]:
# Outlier Detection using Z-score
z_scores = train_data[num_cols].apply(zscore)

plt.figure(figsize=(12, 6))
train_data[num_cols].boxplot(rot=90)
plt.title("Box Plot for Outlier Detection")
plt.show()

# Filter out outliers
train_cleaned= train_data[(z_scores.abs() < 3).all(axis=1)]

1. **Presence of Extreme Outliers:**  
   - A significant outlier is visible in one of the features, with a value exceeding **4 billion**. This extreme deviation indicates a potential data recording error or an abnormally large value that may need further investigation.
   - The rest of the features exhibit relatively stable distributions, with some showing minor outliers.

2. **Features with Outliers:**  
   - Certain numerical features have values that deviate considerably from the interquartile range (IQR), meaning they lie far beyond the typical range of values.
   - Features related to **PrimaryDisplayResolutionHorizontal** and **PrimaryDisplayResolutionVertical** seem to have extreme values, which could be due to incorrect data entry or rare occurrences of high-resolution displays.

3. **Possible Causes of Outliers:**  
   - **Data Entry Errors:** Some values may have been mistakenly recorded at an extremely high magnitude.
   - **Skewed Distributions:** Some features might naturally have long-tailed distributions, leading to values that are extreme but valid.
   - **Device-Specific Characteristics:** If certain devices have unique specifications, they may appear as outliers when compared to more common configurations.


# **Preprocessing**

In [ ]:
#defining features and target
X= train_data.drop(columns=['MachineID', 'target'], errors='ignore')
y= train_data['target']
x_test= test_data.drop(columns=['MachineID'], errors='ignore')

#separating categorical and numerical columns
cat= X.select_dtypes(include=["object"]).columns
num= X.select_dtypes(include=["int64", "float64"]).columns

#Preprocessing pipeline
preprocessor = Pipeline([
    ('num_imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

#Encode categorical variables
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[cat] = encoder.fit_transform(X[cat].astype(str))
x_test[cat] = encoder.transform(x_test[cat].astype(str))

#Apply pipeline transformations
X[num] = preprocessor.fit_transform(X[num])
x_test[num] = preprocessor.transform(x_test[num])

#splitting data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# **Model Training**

In [ ]:
models= {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SGD Classifier': SGDClassifier(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'LightGBM': LGBMClassifier(random_state=42)
}

param_grids = {
    'SGD Classifier': {
        'loss': ['hinge', 'log'],
        'penalty': ['l2', 'l1', 'elasticnet'],
        'alpha': [0.0001, 0.001, 0.01],
    },
    'XGBoost': {
        'n_estimators': [100, 300],
        'learning_rate': [0.01, 0.1],
        'max_depth': [3, 5],
        'subsample': [0.8, 1.0],
        'tree_method': ['hist']
    },
    'LightGBM': {
        'n_estimators': [100, 300],
        'learning_rate': [0.01, 0.1],
        'num_leaves': [20, 31],
        'boosting_type': ['gbdt']
    }
}

best_models = {}
best_scores = {}

for name, model in models.items():
    if name in param_grids:
        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grids[name],
            cv=3,
            scoring='accuracy',
            n_jobs=-1,
            random_state=42
        )
        search.fit(X_train, y_train)
        best_models[name] = search.best_estimator_
    else:
        model.fit(X_train, y_train)
        best_models[name] = model

    y_pred = best_models[name].predict(X_test)
    best_scores[name] = accuracy_score(y_test, y_pred)

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(y_test), yticklabels=np.unique(y_test))
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix - {name}')
    plt.show()

    # Print classification report
    print(f"Classification Report for {name}:\n", classification_report(y_test, y_pred))


# Find the best model
best_model_name = max(best_scores, key=best_scores.get)
best_model_score = best_scores[best_model_name]

In [ ]:
"""#defining models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SGD Classifier': SGDClassifier(max_iter=1000, random_state=42),
    'Random Forest Classifier': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'LightGBM': LGBMClassifier(random_state=42)
}

# Hyperparameter tuning grids
param_grids = {
    'SGD Classifier': {
        'loss': ['hinge', 'log'],
        'penalty': ['l2', 'l1', 'elasticnet'],
        'alpha': [0.0001, 0.001, 0.01]
    },
    'XGBoost': {
        'n_estimators': [100, 300],
        'learning_rate': [0.01, 0.1],
        'max_depth': [3, 5],
        'subsample': [0.8, 1.0],
        'tree_method': ['hist']
    },
    'LightGBM': {
        'n_estimators': [100, 300],
        'learning_rate': [0.01, 0.1],
        'num_leaves': [20, 31],
        'boosting_type': ['gbdt']
    }
}

#training and evaluating models
best_models= {}
best_scores={}

for name, model in models.items():
    if name in param_grids:
        model= RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grids[name],
            cv=3,
            scoring='accuracy',
            n_jobs=-1,
            random_state=42
        )
        model.fit(X_train, y_train)
        best_models[name] = model.best_estimator_
    else:
        model.fit(X_train, y_train)
        best_models[name] = model

    y_pred = best_models[name].predict(X_test)
    best_scores[name] = accuracy_score(y_test, y_pred)

    #confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix - {name}')
    plt.show()

    print(f"Classification Report for {name}:\n", classification_report(y_test, y_pred))

# Find the best model
best_model_name = max(best_scores, key=best_scores.get)
best_model = models[best_model_name]"""

In [ ]:
y_pred_test = best_models[best_model_name].predict(x_test)

sub = pd.DataFrame({'id': range(len(y_pred_test)), 'target': y_pred_test})
sub.to_csv('submission.csv', index=False)